# CSPICE Validation

In [ ]:
import os
import pylupnt as pnt
import spiceypy as sp
import numpy as np

np.set_printoptions(precision=12, linewidth=100, suppress=False)

In [ ]:
kernel_dir = pnt.get_cspice_kernel_dir()
sp.furnsh(os.path.join(kernel_dir, "de440.bsp"))
sp.furnsh(os.path.join(kernel_dir, "naif0012.tls"))

## Time Conversions

In [43]:
for year in (2000, 2025, 2050):
    date = f"{year} May 21, 01:02:03 UTC"
    t_tdb = sp.str2et(date)
    for time in ("UTC", "TT", "TDB"):
        line = time.ljust(3) + " "
        line += sp.timout(t_tdb, f"YYYY-MM-DDTHR:MN:SC.###### ::{time}") + " "
        if time == "UTC":
            line += "0"
        else:
            line += str(sp.unitim(t_tdb, "TDB", time))
        print(line)
        

UTC 2000-05-21T01:02:03.000000 0
TT  2000-05-21T01:03:07.184000 12142987.184
TDB 2000-05-21T01:03:07.185136 12142987.185136192
UTC 2025-05-21T01:02:03.000000 0
TT  2025-05-21T01:03:12.184000 801061392.184
TDB 2025-05-21T01:03:12.185146 801061392.1851462
UTC 2050-05-21T01:02:03.000000 0
TT  2050-05-21T01:03:12.184000 1589979792.184
TDB 2050-05-21T01:03:12.185156 1589979792.185156


## Body Positions

In [81]:
t_utc = pnt.gregorian2time(2020, 5, 21, 1, 2, 3.0)
t_tai = pnt.convert_time(t_utc, pnt.UTC, pnt.TAI)
t_tdb = pnt.convert_time(t_utc, pnt.UTC, pnt.TDB)
locations = ("SSB", "SUN", "EMB", "EARTH", "MOON", "MARS_BARYCENTER")
width = max(len(x) for x in locations) + 1
with open("body_pos_vel_cspice.txt", "w") as f:
    for center in locations:
        for target in locations:
            if center != target:
                rv, lt = sp.spkezr(target, t_tdb, "J2000", "NONE", center)
                line = center.ljust(width) + target.ljust(width) + f"{lt:.9e} "
                line += " ".join(f"{x:+.9e}" for x in rv)
                f.write(line + "\n")
                print(line)

SSB             SUN             4.306222834e+00 -7.419872695e+05 +9.659292737e+05 +4.278400954e+05 -1.408381994e-02 -5.627820863e-03 -2.001044848e-03
SSB             EMB             5.032556135e+02 -7.604591656e+07 -1.195584954e+08 -5.181907715e+07 +2.534386321e+01 -1.370093397e+01 -5.938948538e+00
SSB             EARTH           5.032710330e+02 -7.604958762e+07 -1.195615714e+08 -5.182005163e+07 +2.535189040e+01 -1.370869149e+01 -5.943122540e+00
SSB             MOON            5.020021891e+02 -7.574745701e+07 -1.193084156e+08 -5.173985179e+07 +2.469124806e+01 -1.307024395e+01 -5.599599808e+00
SSB             MARS_BARYCENTER 7.038906758e+02 +7.098457857e+07 -1.798830097e+08 -8.445829605e+07 +2.370522429e+01 +9.564791120e+00 +3.747645442e+00
SUN             SSB             4.306222834e+00 +7.419872695e+05 -9.659292737e+05 -4.278400954e+05 +1.408381994e-02 +5.627820863e-03 +2.001044848e-03
SUN             EMB             5.050667010e+02 -7.530392929e+07 -1.205244247e+08 -5.224691725e+07 +